# Procesamiento ETL 2020 - Estadísticas Vitales

Este notebook se encarga de importar, limpiar, manejar nulos, normalizar y guardar de forma eficiente (en formato Parquet) los datasets correspondientes al año 2020 para:
- Nacimientos
- Muertes Fetales
- Muertes No Fetales

In [75]:
import os
import sys
import pandas as pd

# Agregar la carpeta principal al path para poder importar módulos propios
sys.path.append(os.path.abspath('..'))

from etl.cleaning import *
from etl.nulls import *
from etl.normalization import *
from etl.schema import *

# Asegurarnos de que directorios de salida existan
os.makedirs('../data/processed/nacimientos', exist_ok=True)
os.makedirs('../data/processed/fetales', exist_ok=True)
os.makedirs('../data/processed/no_fetales', exist_ok=True)

## 1. Nacimientos 2020

### Carga del archivo

In [76]:
rutaDatos = '../data/raw/nac2020.csv'
try:
    df_nac = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nac.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 629402 entries, 0 to 629401
Data columns (total 37 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   COD_DPTO    629402 non-null  int64  
 1   COD_MUNIC   629402 non-null  int64  
 2   AREANAC     629402 non-null  int64  
 3   SIT_PARTO   629402 non-null  int64  
 4   OTRO_SIT    1233 non-null    object 
 5   SEXO        629402 non-null  int64  
 6   PESO_NAC    629402 non-null  int64  
 7   TALLA_NAC   629402 non-null  int64  
 8   ANO         629402 non-null  int64  
 9   MES         629402 non-null  int64  
 10  ATEN_PAR    629402 non-null  int64  
 11  T_GES       629402 non-null  int64  
 12  NUMCONSUL   629402 non-null  int64  
 13  TIPO_PARTO  629402 non-null  int64  
 14  MUL_PARTO   629402 non-null  int64  
 15  APGAR1      629402 non-null  int64  
 16  APGAR2      629402 non-null  int64  
 17  IDHEMOCLAS  629402 non-null  int64  
 18  IDFACTORRH  629402 non

### Eliminación de duplicados

In [77]:
encontrar_duplicados(df_nac)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 629402
Filas duplicadas (incluyendo repetidas): 3071
Duplicados únicos a eliminar: 1937


,COD_DPTO,COD_MUNIC,AREANAC,SIT_PARTO,OTRO_SIT,SEXO,PESO_NAC,TALLA_NAC,ANO,MES,...,AREA_RES,N_HIJOSV,FECHA_NACM,N_EMB,SEG_SOCIAL,IDCLASADMI,EDAD_PADRE,NIV_EDUP,ULTCURPAD,PROFESION
5,20,1,3,2,NaN,2,9,9,2020,9,...,3.0,1,NaN,1,2,2.0,999,99,99,5
8,44,560,3,2,NaN,2,9,9,2020,9,...,3.0,1,NaN,1,2,2.0,26,99,99,5
22,20,570,3,2,NaN,2,9,9,2020,3,...,3.0,1,NaN,1,2,2.0,999,99,99,5
75,44,560,3,2,NaN,2,9,9,2020,9,...,3.0,1,NaN,1,2,2.0,36,99,99,5
479,20,45,3,2,NaN,1,9,9,2020,2,...,3.0,1,NaN,1,2,2.0,999,99,99,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629181,27,1,1,2,NaN,1,9,9,2020,4,...,2.0,1,NaN,1,5,NaN,999,99,99,5
629203,27,413,3,2,NaN,1,9,9,2020,5,...,3.0,1,NaN,1,2,2.0,999,99,99,5
629208,27,413,3,2,NaN,1,9,9,2020,6,...,3.0,1,NaN,1,2,2.0,999,99,99,5
629212,27,413,3,2,NaN,1,9,9,2020,5,...,3.0,1,NaN,1,2,2.0,999,99,99,5


In [78]:
df_nac = eliminar_duplicados(df_nac)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 1937
Registros eliminados: 1937
Total final: 627465


### Eliminación de columnas que no son necesarias para el estudio

In [79]:
cols_a_borrar_nac = ['IDPERTET','OTRO_SIT','APGAR1','APGAR2','IDPERTE','ULTCURMAD','H_HIJOSV','FECHA_NACM','N_EMB','ULTCURPAD']
df_nac = eliminar_columnas(df_nac, cols_a_borrar_nac)

Borradas 8 columnas de la lista especificada.


### Estandarización de Nombres de las columnas

In [80]:
# 3. Estandarizar Nombres de Columnas
df_nac = estandarizar_nombres_columnas(df_nac)

### Nueva columna

In [81]:
df_nac["tipo_evento"] = "NAC"

### Manejo de nulos 

In [82]:
resumen_nulos = analizar_nulos(df_nac)

===== ANÁLISIS DE NULOS =====
Total de registros: 627465
Columnas con nulos: 4



,nulos,porcentaje (%)
idclasadmi,71926,11.462950
codptore,9784,1.559290
codmunre,9784,1.559290
area_res,9773,1.557537


Al realizar el análisis de valores nulos, se identificó que las variables codptore, codmunre y area_res presentan una alta correlación en sus valores faltantes, es decir, los registros con valores nulos coinciden en las mismas filas. Esto indica la existencia de datos incompletos en el componente geográfico, por lo que se decidió eliminar dichos registros para garantizar la consistencia del análisis territorial.

Por otro lado, para la variable idclasadmi, se definió una estrategia de imputación en lugar de eliminación, debido a que presenta un porcentaje significativo de valores nulos (~11.46%). Eliminar estos registros implicaría una pérdida considerable de información. Además, al tratarse de una variable categórica asociada al régimen de salud, su ausencia no impide el análisis principal, por lo que se optó por reemplazar los valores faltantes con la categoría “DESCONOCIDO”, preservando así la integridad del dataset.

In [83]:
df_nac[df_nac["codptore"].isnull()].shape


(9784, 30)

In [84]:
df_nac[df_nac["codmunre"].isnull()].shape

(9784, 30)

In [85]:
df_nac[df_nac["codmunre"].isnull()].shape



(9784, 30)

In [86]:
df_nac[df_nac["area_res"].isnull()].shape

(9773, 30)

In [87]:
# 2. eliminar nulos geográficos
df_nac = df_nac.dropna(subset=["codptore", "codmunre", "area_res"])

In [88]:
# 3. imputar otros nulos
estrategia = {
    "idclasadmi": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nac = manejar_nulos(df_nac, estrategia)

In [89]:
resumen_nulos = analizar_nulos(df_nac)

===== ANÁLISIS DE NULOS =====
Total de registros: 617681
Columnas con nulos: 0



,nulos,porcentaje (%)


### Ajustar Tipos de Datos (Bajar peso en memoria)

In [90]:
df_nac = ajustar_tipos_datos(df_nac)

### Verficación de columnas

In [91]:
VALID_COLUMNS_NACIMIENTOS = {
    col: str(df_nac[col].dtype) for col in VALID_COLUMNS_NACIMIENTOS
}

validar_schema(df_nac, VALID_COLUMNS_NACIMIENTOS)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 30


## 2. Muertes Fetales 2020

### Carga de dataset

In [92]:
rutaDatos = '../data/raw/fetal2020.csv'
try:
    df_fet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_fet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33327 entries, 0 to 33326
Data columns (total 43 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   COD_DPTO     33327 non-null  int64  
 1   COD_MUNIC    33327 non-null  int64  
 2   A_DEFUN      33327 non-null  int64  
 3   SIT_DEFUN    33327 non-null  int64  
 4   OTRSITIODE   74 non-null     object 
 5   TIPO_DEFUN   33327 non-null  int64  
 6   ANO          33327 non-null  int64  
 7   MES          33327 non-null  int64  
 8   HORA         33327 non-null  int64  
 9   MINUTOS      33327 non-null  int64  
 10  SEXO         33327 non-null  int64  
 11  CODPRES      33198 non-null  float64
 12  CODPTORE     32642 non-null  float64
 13  CODMUNRE     32642 non-null  float64
 14  AREA_RES     32643 non-null  float64
 15  SEG_SOCIAL   33327 non-null  int64  
 16  IDADMISALUD  28904 non-null  float64
 17  P_PMAN_IRIS  4864 non-null   float64
 18  CONS_EXP     33327 non-n

C:\Users\edwar\AppData\Local\Temp\ipykernel_380076\2290250455.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_fet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')


### Eliminación de duplicados

En el conjunto de datos de muertes fetales se identificaron 369 registros duplicados, lo que representa aproximadamente el 1.10% del total. Este porcentaje se considera bajo, por lo que se procedió a su eliminación sin impacto significativo en la integridad del análisis.

In [93]:
encontrar_duplicados(df_fet)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 33327
Filas duplicadas (incluyendo repetidas): 686
Duplicados únicos a eliminar: 369


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL
33,11,1,1,1,NaN,1,2020,2,0,0,...,1.0,1.0,NaN,NaN,1,P018,P018,402,1,80
81,76,1,1,1,NaN,1,2020,3,0,0,...,1.0,NaN,NaN,NaN,1,P018*P018,P018,402,1,80
143,52,835,1,3,NaN,1,2020,2,0,0,...,NaN,NaN,1.0,NaN,2,P018,P018,402,1,80
145,11,1,1,1,NaN,1,2020,7,0,0,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
192,41,1,1,1,NaN,1,2020,4,0,0,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33291,11,1,1,1,NaN,1,2020,1,0,0,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
33293,11,1,1,1,NaN,1,2020,1,0,0,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80
33294,76,1,1,1,NaN,1,2020,1,0,0,...,1.0,1.0,NaN,NaN,1,P018*P018,P018,402,1,80
33303,54,1,1,1,NaN,1,2020,1,0,0,...,1.0,NaN,NaN,NaN,1,P018,P018,402,1,80


In [94]:
df_fet = eliminar_duplicados(df_fet)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 369
Registros eliminados: 369
Total final: 32958


### Eliminación de columnas que no son necesarias para el estudio

In [95]:
cols_a_borrar_fet = ['HORA','MINUTOS','OTRSITIODE','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE','CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER']
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet)

Borradas 18 columnas de la lista especificada.


### Estandarizar Nombres de Columnas

In [96]:
df_fet = estandarizar_nombres_columnas(df_fet)

### Nueva Columna 

In [97]:
df_fet["tipo_evento"] = "FETAL"

### Manejo de nulos

In [98]:
resumen_nulos = analizar_nulos(df_fet)

===== ANÁLISIS DE NULOS =====
Total de registros: 32958
Columnas con nulos: 4



,nulos,porcentaje (%)
idadmisalud,4394,13.332120
codmunre,679,2.060198
codptore,679,2.060198
area_res,678,2.057164


In [99]:
df_fet[df_fet["codptore"].isnull()].shape


(679, 26)

In [100]:
df_fet[df_fet["codmunre"].isnull()].shape

(679, 26)

In [101]:
df_fet[df_fet["area_res"].isnull()].shape

(678, 26)

Se observó que los valores nulos en las variables geográficas (codptore y codmunre) coinciden exactamente en los mismos registros, mientras que la variable area_res presenta una ligera variación de un registro adicional. Esto confirma la presencia de datos incompletos en el componente geográfico. Por tanto, se decidió eliminar dichos registros para garantizar la coherencia y calidad del análisis espacial.

In [102]:
df_fet = df_fet.dropna(subset=["codptore", "codmunre", "area_res"])

In [103]:
estrategia = {
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_fet = manejar_nulos(df_fet, estrategia)

### Ajustar Tipos de Datos (Bajar peso en memoria)

In [104]:
df_fet = ajustar_tipos_datos(df_fet)

### Verficación de columnas

In [105]:
VALID_COLUMNS_FETALES = {
    col: str(df_fet[col].dtype) for col in VALID_COLUMNS_FETALES
}

validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 24

➕ Columnas extra (2): ['seg_social', 'idadmisalud']


Podemos observar que tenemso 'seg_social', 'idadmisalud' extras en df_fet, las vamos a borrar.
Debido a que en el 2024 estos datos no se encuentran disponibles, no tiene sentido mantenerlos en el esquema. Además, esto nos permitirá tener un esquema consistente a lo largo de los años, facilitando el análisis comparativo.

In [106]:
cols_a_borrar_fet_extra = ['seg_social', 'idadmisalud']
df_fet = eliminar_columnas(df_fet, cols_a_borrar_fet_extra)

Borradas 2 columnas de la lista especificada.


In [107]:
validar_schema(df_fet, VALID_COLUMNS_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 24


## 3. Muertes No Fetales 2020

### Carga de Dataset

In [108]:
rutaDatos = '../data/raw/nofetal2020.csv'
try:
    df_nofet = pd.read_csv(rutaDatos, sep=',', encoding='latin-1')
    print(f"Archivo encontrado: \n")
    df_nofet.info()
except FileNotFoundError:
    print(f"Archivo no encontrado: {rutaDatos}")

Archivo encontrado: 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300853 entries, 0 to 300852
Data columns (total 55 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   COD_DPTO     300853 non-null  int64  
 1   COD_MUNIC    300853 non-null  int64  
 2   A_DEFUN      300853 non-null  int64  
 3   SIT_DEFUN    300853 non-null  int64  
 4   OTRSITIODE   6431 non-null    object 
 5   TIPO_DEFUN   300853 non-null  int64  
 6   ANO          300853 non-null  int64  
 7   MES          300853 non-null  int64  
 8   HORA         300853 non-null  int64  
 9   MINUTOS      300853 non-null  int64  
 10  SEXO         300853 non-null  int64  
 11  EST_CIVIL    300853 non-null  int64  
 12  GRU_ED1      300853 non-null  int64  
 13  GRU_ED2      300853 non-null  int64  
 14  NIVEL_EDU    300853 non-null  int64  
 15  ULTCURFAL    300853 non-null  int64  
 16  MUERTEPORO   293231 non-null  float64
 17  SIMUERTEPO   471 non-null     float64
 18  OC

### Eliminar duplicados

In [109]:
encontrar_duplicados(df_nofet)

===== ANÁLISIS DE DUPLICADOS =====
Total de registros: 300853
Filas duplicadas (incluyendo repetidas): 81
Duplicados únicos a eliminar: 42


,COD_DPTO,COD_MUNIC,A_DEFUN,SIT_DEFUN,OTRSITIODE,TIPO_DEFUN,ANO,MES,HORA,MINUTOS,...,C_MUERTEB,C_MUERTEC,C_MUERTED,C_MUERTEE,ASIS_MED,CAUSA_MULT,C_BAS1,CAUSA_667,IDPROFCER,CAU_HOMOL
10163,23,466,3,5,NaN,2,2020,8,19,30,...,NaN,NaN,NaN,NaN,2,T794/S269/T141 X95,X954,512,1,101
10595,5,234,9,9,NaN,2,2020,2,0,0,...,NaN,NaN,NaN,NaN,2,T76/T76 Y34/T76 Y34,Y349,513,1,102
30202,11,1,1,3,NaN,2,2020,10,0,0,...,NaN,NaN,NaN,NaN,2,G931/T71/T71 X70,X700,511,1,100
31637,19,142,3,6,AREA DE CULTIVO,2,2020,12,0,0,...,NaN,NaN,NaN,NaN,2,S066/S062/S019 X95,X958,512,1,101
33778,19,100,3,5,NaN,2,2020,9,0,0,...,NaN,NaN,NaN,NaN,2,S062/S097 S029/T141 X95,X954,512,1,101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272914,70,429,1,3,NaN,2,2020,8,12,0,...,NaN,NaN,1.0,NaN,2,R98,R98,0,1,89
280139,5,1,1,1,NaN,2,2020,11,11,0,...,1.0,NaN,NaN,NaN,1,J960/B24*A169,B200,107,1,9
282764,13,1,1,1,NaN,2,2020,12,17,33,...,1.0,NaN,NaN,NaN,1,P291/P072,P072,403,1,81
291269,13,1,1,1,NaN,2,2020,12,17,33,...,1.0,NaN,NaN,NaN,1,P291/P072,P072,403,1,81


In [110]:
df_nofet = eliminar_duplicados(df_nofet)

===== LIMPIEZA DE DUPLICADOS =====
Duplicados detectados: 42
Registros eliminados: 42
Total final: 300811


### Eliminación de columnas que no nos sirven para el estudio 

In [111]:
cols_a_borrar_nofet = ['OCUPACION','HORA','MINUTOS','OTRSITIODE','CODPRES','P_PMAN_IRIS','CONS_EXP','ULTCURMAD','CODOCUR','CODMUNOC','C_MUERTE','C_MUERTEB','C_MUERTEC','C_MUERTED','C_MUERTEE','CAUSA_MULT','C_BAS1','CAUSA_667','IDPROFCER','MU_PARTO','CAU_HOMOL']
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_nofet)

Borradas 21 columnas de la lista especificada.


### Estandarizar nombre de columnas

In [112]:
df_nofet = estandarizar_nombres_columnas(df_nofet)

### Nueva columna

In [113]:
df_nofet["tipo_evento"] = "NOFETAL"

### Manejo de nulos
El conjunto de datos de muertes no fetales presenta un alto porcentaje de valores nulos en múltiples variables, superando el 90% en varios casos. Esto sugiere que dichas variables no son aplicables o no fueron diligenciadas en este tipo de registros. Por esta razón, se optó por eliminar estas columnas para evitar ruido en el análisis.

Adicionalmente, las variables geográficas presentaron un bajo porcentaje de valores nulos (~0.5%), los cuales fueron eliminados para garantizar la consistencia espacial. Finalmente, las variables con porcentajes moderados de valores faltantes fueron tratadas mediante imputación, con el fin de preservar la mayor cantidad de información posible.

In [114]:
resumen_nulos = analizar_nulos(df_nofet)

===== ANÁLISIS DE NULOS =====
Total de registros: 300811
Columnas con nulos: 18



,nulos,porcentaje (%)
simuertepo,300340,99.843423
t_ges,294446,97.884053
niv_edum,294445,97.883721
n_hijosm,294445,97.883721
peso_nac,294445,97.883721
tipo_emb,294445,97.883721
est_civm,294445,97.883721
t_parto,294445,97.883721
edad_madre,294445,97.883721
n_hijosv,294444,97.883389


In [115]:
#  eliminar nulos geográficos
df_nofet = df_nofet.dropna(subset=["codptore", "codmunre", "area_res"])

In [116]:
estrategia = {
    "ocupacion": {"metodo": "fill", "valor": "DESCONOCIDO"},
    "muerteporo": {"metodo": "fill_mode"},
    "idadmisalud": {"metodo": "fill", "valor": "DESCONOCIDO"}
}

df_nofet = manejar_nulos(df_nofet, estrategia)

In [117]:
columnas_eliminar = [
    "simuertepo",
    "t_ges",
    "niv_edum",
    "n_hijosm",
    "tipo_emb",
    "t_parto",
    "est_civm",
    "peso_nac",
    "edad_madre",
    "n_hijosv",
    "emb_mes",
    "emb_sem",
    "emb_fal"
]

In [118]:
# eliminar columnas inútiles (>90% nulos)
df_nofet = df_nofet.drop(columns=columnas_eliminar)

### Ajustar Tipos de Datos (Bajar peso en memoria)

In [119]:
df_nofet = ajustar_tipos_datos(df_nofet)

### Verficación de columnas

In [120]:
VALID_COLUMNS_NO_FETALES = {
    col: str(df_nofet[col].dtype) for col in VALID_COLUMNS_NO_FETALES
}

validar_schema(df_nofet, VALID_COLUMNS_NO_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 21

➕ Columnas extra (1): ['idadmisalud']


Podemos observar que tenemso 'idadmisalud' extra en df_nofet, las vamos a borrar, porque no esta no aparece en todos los años.

In [121]:
cols_a_borrar_fet_extra = ['idadmisalud']
df_nofet = eliminar_columnas(df_nofet, cols_a_borrar_fet_extra)

Borradas 1 columnas de la lista especificada.


In [122]:
validar_schema(df_nofet, VALID_COLUMNS_NO_FETALES)


=== RESULTADO VALIDACIÓN ===
✔ Columnas correctas: 21


## Guardar datos procesados

In [ ]:
AÑO = 2020

out_nac = f"../data/processed/nacimientos/nac{AÑO}.parquet"
out_fet = f"../data/processed/fetales/fet{AÑO}.parquet"
out_nofet = f"../data/processed/no_fetales/nofet{AÑO}.parquet"

df_nac.to_parquet(out_nac, index=False)
df_fet.to_parquet(out_fet, index=False)
df_nofet.to_parquet(out_nofet, index=False)

print(f"Guardado: {out_nac}")
print(f"Guardado: {out_fet}")
print(f"Guardado: {out_nofet}")

Guardado: ../data/processed/nacimientos/nac2020.parquet
Guardado: ../data/processed/fetales/fet2020.parquet
Guardado: ../data/processed/no_fetales/nofet2020.parquet
